In [ ]:
!pip -q install mne mne-bids scikit-learn tensorflow joblib

In [ ]:
import os
import glob
import zipfile
import warnings
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    auc
)

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping

import joblib

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")

print("MNE:", mne.__version__)
print("TensorFlow:", tf.__version__)

In [ ]:
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed:", SEED)

In [ ]:
PROJECT_NAME = "P300 EEG-Based BCI"

DATASET_ID = "nm000351"

N_SUBJECTS = 5

TARGET_CODE = 2
NONTARGET_CODE = 1

LOW_FREQ = 0.1
HIGH_FREQ = 30.0

NOTCH_FREQ = 60.0

TMIN = 0.0
TMAX = 1.0

DOWNSAMPLE_FACTOR = 4

print("Project:", PROJECT_NAME)
print("Dataset:", DATASET_ID)
print("Subjects:", N_SUBJECTS)
print("Target:", TARGET_CODE)
print("Non-target:", NONTARGET_CODE)

In [ ]:
from google.colab import files

print("Select eggtest.zip")

uploaded = files.upload()

print("\nUpload completed.")

for filename in uploaded:
    print(filename)

Select eggtest.zip


In [ ]:
import os

print("Files in /content:\n")

for item in os.listdir("/content"):
    print(item)

In [ ]:
import glob

zip_files = glob.glob("/content/*.zip")

print("ZIP files found:")

for f in zip_files:
    print(f)

if len(zip_files) == 0:
    raise FileNotFoundError(
        "❌ No ZIP file found. Please upload eggtest.zip."
    )

ZIP_PATH = "/content/eggtest.zip"

if not os.path.exists(ZIP_PATH):
    ZIP_PATH = zip_files[0]

print("\nUsing ZIP:")
print(ZIP_PATH)

In [ ]:
import zipfile
import os

EXTRACT_PATH = "/content/bigP3BCI"

os.makedirs(EXTRACT_PATH, exist_ok=True)

print("Extracting eggtest.zip...")
print("Please wait...")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("✅ Extraction completed!")

In [ ]:
print("Extracted contents:\n")

for item in os.listdir("/content/bigP3BCI"):
    print(item)

In [ ]:
import os

print("Searching for subjects...\n")

for root, dirs, files in os.walk("/content/bigP3BCI"):
    for d in dirs:
        if d.lower().startswith("sub-"):
            print(os.path.join(root, d))

In [ ]:
# CELL 12 — Find ALL BDF files robustly

import os

DATA_ROOT = "/content/bigP3BCI"

eeg_files = []

for root, dirs, files in os.walk(DATA_ROOT):
    for file in files:
        if file.lower().endswith(".bdf"):
            eeg_files.append(os.path.join(root, file))

eeg_files = sorted(eeg_files)

print("DATA ROOT:")
print(DATA_ROOT)

print("\nTotal BDF files found:", len(eeg_files))

if len(eeg_files) == 0:
    print("\n❌ No BDF files found.")
    print("\nChecking whether the dataset folder exists:")
    print(os.path.exists(DATA_ROOT))
else:
    print("\n✅ BDF files found:\n")
    for file in eeg_files[:20]:
        print(file)

In [ ]:
# CELL 13 — Display BDF files by subject

from collections import defaultdict

subject_files = defaultdict(list)

for file in eeg_files:
    parts = file.split(os.sep)

    # Find the subject folder
    subject = next(
        (p for p in parts if p.startswith("sub-")),
        None
    )

    if subject is not None:
        subject_files[subject].append(file)

print("EEG files by subject:\n")

for subject in sorted(subject_files):
    print(f"{subject}: {len(subject_files[subject])} BDF files")

    for file in subject_files[subject][:5]:
        print("   ", os.path.basename(file))

    if len(subject_files[subject]) > 5:
        print("    ...")

    print()

In [ ]:
# CELL 14 — BDF EEG loader

import mne

def load_raw_bdf(file_path):
    print("Loading:")
    print(file_path)

    raw = mne.io.read_raw_bdf(
        file_path,
        preload=True,
        verbose=False
    )

    return raw

print("✅ BDF loader is ready.")

In [ ]:
# CELL 15 — Load one BDF EEG recording

import mne

if not eeg_files:
    raise RuntimeError(
        "❌ eeg_files is empty. Run Cell 12 again."
    )

first_file = eeg_files[0]

print("Loading EEG file:")
print(first_file)
print("\nPlease wait...")

raw = mne.io.read_raw_bdf(
    first_file,
    preload=True,
    verbose=False
)

print("\n✅ EEG loaded successfully!")
print(raw)

In [ ]:
def load_raw_eeg(file_path):

    extension = os.path.splitext(file_path)[1].lower()

    print("\nLoading:")
    print(file_path)

    if extension == ".vhdr":

        raw = mne.io.read_raw_brainvision(
            file_path,
            preload=True,
            verbose=False
        )

    elif extension == ".edf":

        raw = mne.io.read_raw_edf(
            file_path,
            preload=True,
            verbose=False
        )

    elif extension == ".bdf":

        raw = mne.io.read_raw_bdf(
            file_path,
            preload=True,
            verbose=False
        )

    elif extension == ".set":

        raw = mne.io.read_raw_eeglab(
            file_path,
            preload=True,
            verbose=False
        )

    elif extension == ".fif":

        raw = mne.io.read_raw_fif(
            file_path,
            preload=True,
            verbose=False
        )

    else:

        raise ValueError(
            "Unsupported EEG format: " + extension
        )

    return raw

In [ ]:
# CELL 17 — Select EEG channels safely

print("Total channels:", len(raw.ch_names))
print("\nChannel names:")
print(raw.ch_names)

# Find EEG channels automatically
eeg_picks = mne.pick_types(
    raw.info,
    eeg=True,
    exclude=[]
)

print("\nNumber of EEG channels found:", len(eeg_picks))

if len(eeg_picks) == 0:
    raise RuntimeError(
        "❌ No EEG channels were detected. "
        "Please check the channel types in Cell 16."
    )

# Keep only EEG channels
raw_eeg = raw.copy().pick(eeg_picks)

print("\n✅ EEG channels selected successfully!")
print("Number of EEG channels:", len(raw_eeg.ch_names))
print("\nSelected channels:")
print(raw_eeg.ch_names)

In [ ]:
print("Number of channels:", len(raw.ch_names))

print("\nChannels:")

for i, channel in enumerate(raw.ch_names):

    print(i, channel)

In [ ]:
picks = mne.pick_types(
    raw.info,
    eeg=True,
    eog=False,
    stim=False,
    exclude="bads"
)

raw.pick(picks)

print("EEG channels after selection:")
print(len(raw.ch_names))

print(raw.ch_names)

In [ ]:
try:

    montage = mne.channels.make_standard_montage(
        "standard_1020"
    )

    raw.set_montage(
        montage,
        on_missing="ignore"
    )

    print("✅ Standard 10-20 montage applied.")

except Exception as e:

    print("Montage warning:")
    print(e)

In [ ]:
raw.filter(
    l_freq=LOW_FREQ,
    h_freq=HIGH_FREQ,
    verbose=False
)

print(
    f"✅ Band-pass filter applied: "
    f"{LOW_FREQ}-{HIGH_FREQ} Hz"
)

In [ ]:
try:

    raw.notch_filter(
        freqs=NOTCH_FREQ,
        verbose=False
    )

    print(
        f"✅ {NOTCH_FREQ} Hz notch filter applied."
    )

except Exception as e:

    print("Notch filter warning:")
    print(e)

In [ ]:
raw.plot(
    duration=10,
    n_channels=min(10, len(raw.ch_names)),
    scalings="auto"
)

In [ ]:
print("Number of annotations:", len(raw.annotations))

print("\nAnnotations:")

for i, annotation in enumerate(raw.annotations[:50]):

    print(
        i,
        "Onset:",
        annotation["onset"],
        "Duration:",
        annotation["duration"],
        "Description:",
        annotation["description"]
    )

In [ ]:
events, event_id = mne.events_from_annotations(
    raw,
    verbose=False
)

print("Event ID:")
print(event_id)

print("\nNumber of events:", len(events))

In [ ]:
print("Expected Target code:", TARGET_CODE)
print("Expected Non-target code:", NONTARGET_CODE)

print("\nAvailable event codes:")

for name, code in event_id.items():

    print(
        name,
        "->",
        code
    )

In [ ]:
unique_codes, counts = np.unique(
    events[:, 2],
    return_counts=True
)

print("Event counts:\n")

for code, count in zip(
    unique_codes,
    counts
):

    print(
        "Code:",
        code,
        "Count:",
        count
    )

In [ ]:
# DIAGNOSTIC — Check actual P300 events

print("event_id dictionary:")
print(event_id)

print("\nFirst 20 events:")
print(events[:20])

print("\nUnique event codes:")
print(sorted(set(events[:, 2])))

print("\nNumber of events:")
print(len(events))

In [ ]:
# CELL — Inspect the BIDS events.tsv file

import os
import pandas as pd

# Find the events.tsv corresponding to the BDF file
bdf_file = first_file

events_tsv = bdf_file.replace("_eeg.bdf", "_events.tsv")

print("BDF file:")
print(bdf_file)

print("\nEvents TSV:")
print(events_tsv)

if not os.path.exists(events_tsv):
    raise FileNotFoundError(
        "❌ Matching events.tsv file was not found."
    )

# Read the BIDS events file
events_df = pd.read_csv(
    events_tsv,
    sep="\t"
)

print("\n✅ Events TSV loaded!")
print("Shape:", events_df.shape)

print("\nColumns:")
print(events_df.columns.tolist())

print("\nFirst 20 rows:")
display(events_df.head(20))

In [ ]:
# CELL — Inspect event information

for column in events_df.columns:
    print("\n==============================")
    print("COLUMN:", column)
    print("==============================")

    print(events_df[column].unique()[:30])

In [ ]:
# CELL — Find a recording containing BOTH Target and NonTarget events

import os
import pandas as pd

DATA_ROOT = "/content/bigP3BCI"

candidate_runs = []

for root, dirs, files in os.walk(DATA_ROOT):
    for file in files:
        if file.endswith("_events.tsv"):

            events_path = os.path.join(root, file)

            try:
                df = pd.read_csv(events_path, sep="\t")

                if "trial_type" in df.columns:

                    event_types = set(
                        df["trial_type"]
                        .dropna()
                        .astype(str)
                        .str.strip()
                    )

                    if "Target" in event_types and "NonTarget" in event_types:
                        candidate_runs.append(events_path)

            except Exception as e:
                print("Could not read:", events_path)
                print(e)

print("========================================")
print("RUNS WITH TARGET + NONTARGET")
print("========================================")

print("Number of suitable runs:", len(candidate_runs))

for i, path in enumerate(candidate_runs[:20]):
    print(i, path)


if len(candidate_runs) == 0:
    raise RuntimeError(
        "❌ No recording containing both Target and NonTarget was found."
    )

print("\n✅ Suitable recording found!")
print(candidate_runs[0])

In [ ]:
# CELL — Load the selected recording and create events

events_tsv = candidate_runs[0]

# Convert events.tsv path to corresponding BDF path
bdf_file = events_tsv.replace("_events.tsv", "_eeg.bdf")

print("BDF:")
print(bdf_file)

print("\nEvents TSV:")
print(events_tsv)

# Load EEG
raw = mne.io.read_raw_bdf(
    bdf_file,
    preload=True,
    verbose=False
)

# Select EEG channels
eeg_picks = mne.pick_types(
    raw.info,
    eeg=True,
    exclude=[]
)

raw_eeg = raw.copy().pick(eeg_picks)

print("\n✅ EEG loaded")
print("Channels:", len(raw_eeg.ch_names))

# Read BIDS events
events_df = pd.read_csv(
    events_tsv,
    sep="\t"
)

print("\nEvent types:")
print(events_df["trial_type"].value_counts())

# Create MNE events directly from the sample column
events = []

for _, row in events_df.iterrows():

    sample = int(row["sample"])
    trial_type = str(row["trial_type"]).strip()

    if trial_type == "NonTarget":
        code = 1

    elif trial_type == "Target":
        code = 2

    else:
        continue

    events.append([sample, 0, code])

events = np.array(events, dtype=int)

event_id = {
    "NonTarget": 1,
    "Target": 2
}

print("\nMNE event_id:")
print(event_id)

print("\nUnique event codes:")
print(np.unique(events[:, 2]))

print("\nTotal events:", len(events))

In [ ]:
# CELL — Create Target and NonTarget P300 epochs

TMIN = 0.0
TMAX = 0.8

epochs = mne.Epochs(
    raw_eeg,
    events,
    event_id={
        "NonTarget": 1,
        "Target": 2
    },
    tmin=TMIN,
    tmax=TMAX,
    baseline=(0.0, 0.1),
    preload=True,
    reject_by_annotation=True,
    verbose=False
)

print("\n✅ P300 epochs created successfully!")

print(epochs)

print("\nNumber of epochs:", len(epochs))

print("\nEpoch shape:")
print(epochs.get_data().shape)

print("\nMeaning:")
print("(epochs, channels, time samples)")

In [ ]:
X = epochs.get_data()

print("Epoch data shape:")
print(X.shape)

print("\nMeaning:")
print("(epochs, channels, time samples)")

In [ ]:
event_codes = epochs.events[:, 2]

print("Unique event codes:")
print(np.unique(event_codes, return_counts=True))

In [ ]:
target_evoked = epochs["Target"].average()

target_evoked.plot(
    spatial_colors=True
)

plt.show()

In [ ]:
nontarget_evoked = epochs["NonTarget"].average()

nontarget_evoked.plot(
    spatial_colors=True
)

plt.show()

In [ ]:
channel_index = 0

plt.figure(figsize=(10, 5))

plt.plot(
    target_evoked.times,
    target_evoked.data[channel_index] * 1e6,
    label="Target"
)

plt.plot(
    nontarget_evoked.times,
    nontarget_evoked.data[channel_index] * 1e6,
    label="Non-target"
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (µV)")

plt.title(
    "P300 Target vs Non-target"
)

plt.legend()
plt.grid()

plt.show()

In [ ]:
X = epochs.get_data()

event_codes = epochs.events[:, 2]

y = np.where(
    event_codes == TARGET_CODE,
    1,
    0
)

print("X:", X.shape)
print("y:", y.shape)

print("\nLabels:")
print(
    np.unique(
        y,
        return_counts=True
    )
)

In [ ]:
X_downsampled = X[
    :,
    :,
    ::DOWNSAMPLE_FACTOR
]

print("Original:")
print(X.shape)

print("\nAfter downsampling:")
print(X_downsampled.shape)

In [ ]:
X_flat = X_downsampled.reshape(
    X_downsampled.shape[0],
    -1
)

print("Final ML input shape:")
print(X_flat.shape)

In [ ]:
# CELL — Preprocess one subject using BIDS events.tsv

def preprocess_subject(subject_folder):

    subject_name = os.path.basename(subject_folder)

    print("\n")
    print("=" * 80)
    print("PROCESSING:", subject_name)
    print("=" * 80)

    subject_eeg_files = []

    # Find BDF files
    for root, dirs, files in os.walk(subject_folder):

        for filename in files:

            if filename.lower().endswith("_eeg.bdf"):

                subject_eeg_files.append(
                    os.path.join(root, filename)
                )

    subject_eeg_files = sorted(subject_eeg_files)

    print("EEG files:", len(subject_eeg_files))

    if len(subject_eeg_files) == 0:

        print("❌ No EEG BDF files for", subject_name)

        return None, None

    all_X = []
    all_y = []

    # --------------------------------------------------
    # Process every BDF file
    # --------------------------------------------------

    for file_path in subject_eeg_files:

        try:

            print("\nChecking:")
            print(os.path.basename(file_path))

            # Corresponding events.tsv
            events_tsv = file_path.replace(
                "_eeg.bdf",
                "_events.tsv"
            )

            if not os.path.exists(events_tsv):

                print("⚠️ events.tsv not found")
                continue

            # --------------------------------------------------
            # Read events.tsv
            # --------------------------------------------------

            events_df = pd.read_csv(
                events_tsv,
                sep="\t"
            )

            if "trial_type" not in events_df.columns:

                print("⚠️ trial_type column missing")
                continue

            # Get event types
            event_types = set(
                events_df["trial_type"]
                .dropna()
                .astype(str)
                .str.strip()
            )

            print("Event types:", event_types)

            # We need both classes
            if "Target" not in event_types:

                print("⚠️ Target not found")
                continue

            if "NonTarget" not in event_types:

                print("⚠️ NonTarget not found")
                continue

            # --------------------------------------------------
            # Load BDF
            # --------------------------------------------------

            raw = mne.io.read_raw_bdf(
                file_path,
                preload=True,
                verbose=False
            )

            # --------------------------------------------------
            # Select EEG channels
            # --------------------------------------------------

            eeg_picks = mne.pick_types(
                raw.info,
                eeg=True,
                eog=False,
                stim=False,
                exclude="bads"
            )

            if len(eeg_picks) == 0:

                print("⚠️ No EEG channels")
                continue

            raw.pick(eeg_picks)

            # --------------------------------------------------
            # Montage
            # --------------------------------------------------

            try:

                montage = mne.channels.make_standard_montage(
                    "standard_1020"
                )

                raw.set_montage(
                    montage,
                    on_missing="ignore"
                )

            except Exception:

                pass

            # --------------------------------------------------
            # Band-pass filter
            # --------------------------------------------------

            raw.filter(
                LOW_FREQ,
                HIGH_FREQ,
                verbose=False
            )

            # --------------------------------------------------
            # Notch filter
            # --------------------------------------------------

            try:

                raw.notch_filter(
                    NOTCH_FREQ,
                    verbose=False
                )

            except Exception:

                pass

            # --------------------------------------------------
            # Create MNE events from events.tsv
            # --------------------------------------------------

            event_list = []

            for _, row in events_df.iterrows():

                trial_type = str(
                    row["trial_type"]
                ).strip()

                sample = int(
                    row["sample"]
                )

                if trial_type == "NonTarget":

                    code = 1

                elif trial_type == "Target":

                    code = 2

                else:

                    continue

                event_list.append(
                    [sample, 0, code]
                )

            events = np.array(
                event_list,
                dtype=int
            )

            event_id = {
                "NonTarget": 1,
                "Target": 2
            }

            print(
                "Target events:",
                np.sum(events[:, 2] == 2)
            )

            print(
                "NonTarget events:",
                np.sum(events[:, 2] == 1)
            )

            # --------------------------------------------------
            # Create epochs
            # --------------------------------------------------

            epochs = mne.Epochs(
                raw,
                events,
                event_id=event_id,
                tmin=TMIN,
                tmax=TMAX,
                baseline=(0.0, 0.1),
                preload=True,
                reject_by_annotation=True,
                verbose=False
            )

            if len(epochs) == 0:

                print("⚠️ No valid epochs")
                continue

            # --------------------------------------------------
            # Get EEG data
            # --------------------------------------------------

            X_file = epochs.get_data()

            codes = epochs.events[:, 2]

            # Target = 1
            # NonTarget = 0

            y_file = np.where(
                codes == 2,
                1,
                0
            )

            print(
                "Original shape:",
                X_file.shape
            )

            # --------------------------------------------------
            # Downsample
            # --------------------------------------------------

            X_file = X_file[
                :,
                :,
                ::DOWNSAMPLE_FACTOR
            ]

            print(
                "After downsampling:",
                X_file.shape
            )

            # --------------------------------------------------
            # Flatten
            # --------------------------------------------------

            X_file = X_file.reshape(
                X_file.shape[0],
                -1
            )

            print(
                "After flattening:",
                X_file.shape
            )

            # Store
            all_X.append(X_file)
            all_y.append(y_file)

            print(
                "✅ Processed:",
                os.path.basename(file_path)
            )

        except Exception as e:

            print(
                "⚠️ Error:",
                os.path.basename(file_path)
            )

            print(
                "Reason:",
                e
            )

    # --------------------------------------------------
    # Combine all recordings
    # --------------------------------------------------

    if len(all_X) == 0:

        print(
            "\n❌ No usable recordings for",
            subject_name
        )

        return None, None

    X_subject = np.concatenate(
        all_X,
        axis=0
    )

    y_subject = np.concatenate(
        all_y,
        axis=0
    )

    print("\n" + "=" * 80)

    print(
        "FINAL:",
        subject_name
    )

    print(
        "X shape:",
        X_subject.shape
    )

    print(
        "y shape:",
        y_subject.shape
    )

    print(
        "Labels:",
        np.unique(
            y_subject,
            return_counts=True
        )
    )

    print("=" * 80)

    return X_subject, y_subject

In [ ]:
# CELL — Process all subjects and create subject_data

subject_data = {}

# Find subject folders directly from the dataset
selected_subjects = sorted([
    os.path.join(DATA_ROOT, folder)
    for folder in os.listdir(DATA_ROOT)
    if folder.startswith("sub-")
    and os.path.isdir(os.path.join(DATA_ROOT, folder))
])

print("Subjects to process:")
print(selected_subjects)

print("\n" + "=" * 80)
print("PROCESSING ALL SUBJECTS")
print("=" * 80)

for subject_folder in selected_subjects:

    subject_name = os.path.basename(subject_folder)

    X_subject, y_subject = preprocess_subject(
        subject_folder
    )

    if X_subject is not None:

        subject_data[subject_name] = {
            "X": X_subject,
            "y": y_subject
        }

        print(
            f"✅ {subject_name} added to subject_data"
        )

    else:

        print(
            f"⚠️ {subject_name} skipped"
        )


# --------------------------------------------------
# Dataset summary
# --------------------------------------------------

print("\n")
print("=" * 80)
print("DATASET SUMMARY")
print("=" * 80)

if len(subject_data) == 0:

    print("❌ No subjects were successfully processed.")

else:

    for subject, data in subject_data.items():

        print(
            subject,
            "X =",
            data["X"].shape,
            "y =",
            data["y"].shape
        )

    print("\nTotal subjects loaded:")
    print(len(subject_data))

In [ ]:
# CELL — Check loaded subjects

print("Subjects currently stored in subject_data:")

if "subject_data" not in globals():
    print("❌ subject_data does not exist.")
else:
    print(subject_data.keys())

    loaded_subjects = sorted(
        subject_data.keys(),
        key=lambda x: int(x.split("-")[1])
        if x.split("-")[1].isdigit()
        else 999
    )

    print("\nSuccessfully loaded:")
    print(loaded_subjects)

    print("\nNumber of subjects:")
    print(len(loaded_subjects))

In [ ]:
def create_ann(input_size):

    model = Sequential([

        Input(
            shape=(input_size,)
        ),

        Dense(
            256,
            activation="relu"
        ),

        BatchNormalization(),

        Dropout(0.30),

        Dense(
            128,
            activation="relu"
        ),

        BatchNormalization(),

        Dropout(0.30),

        Dense(
            64,
            activation="relu"
        ),

        Dropout(0.20),

        Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
input_size = subject_data[
    loaded_subjects[0]
]["X"].shape[1]

model = create_ann(
    input_size
)

model.summary()

In [ ]:
loso_results = []

all_true = []
all_pred = []
all_prob = []

print("LOSO evaluation initialized.")

In [ ]:
for test_subject in loaded_subjects:

    print("\n")
    print("=" * 90)
    print("TEST SUBJECT:", test_subject)
    print("=" * 90)

    train_subjects = [
        s for s in loaded_subjects
        if s != test_subject
    ]

    print(
        "Training subjects:",
        train_subjects
    )

    # -------------------------
    # Training data
    # -------------------------

    X_train = np.concatenate(
        [
            subject_data[s]["X"]
            for s in train_subjects
        ],
        axis=0
    )

    y_train = np.concatenate(
        [
            subject_data[s]["y"]
            for s in train_subjects
        ],
        axis=0
    )

    # -------------------------
    # Test data
    # -------------------------

    X_test = subject_data[
        test_subject
    ]["X"]

    y_test = subject_data[
        test_subject
    ]["y"]

    print(
        "Training:",
        X_train.shape
    )

    print(
        "Testing :",
        X_test.shape
    )

    # -------------------------
    # Standardization
    # -------------------------

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    )

    X_test_scaled = scaler.transform(
        X_test
    )

    # -------------------------
    # Create model
    # -------------------------

    model = create_ann(
        X_train_scaled.shape[1]
    )

    # -------------------------
    # Early stopping
    # -------------------------

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    # -------------------------
    # Train
    # -------------------------

    history = model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.20,
        epochs=10,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )

    # -------------------------
    # Prediction
    # -------------------------

    probabilities = model.predict(
        X_test_scaled,
        verbose=0
    ).ravel()

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    # -------------------------
    # Metrics
    # -------------------------

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    try:

        roc_auc = roc_auc_score(
            y_test,
            probabilities
        )

    except:

        roc_auc = np.nan

    print("\nRESULTS")
    print("----------------")

    print(
        "Accuracy :",
        accuracy
    )

    print(
        "Precision:",
        precision
    )

    print(
        "Recall   :",
        recall
    )

    print(
        "F1 Score :",
        f1
    )

    print(
        "ROC-AUC  :",
        roc_auc
    )

    # Store results
    loso_results.append({

        "Subject": test_subject,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "ROC_AUC": roc_auc

    })

    all_true.extend(
        y_test
    )

    all_pred.extend(
        predictions
    )

    all_prob.extend(
        probabilities
    )

print("\n")
print("✅ LOSO training completed.")

In [ ]:
results_df = pd.DataFrame(
    loso_results
)

results_df

In [ ]:
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
]

print("MEAN")
print("=" * 50)

print(
    results_df[metrics].mean()
)

print("\nSTANDARD DEVIATION")
print("=" * 50)

print(
    results_df[metrics].std()
)

In [ ]:
all_true = np.array(
    all_true
)

all_pred = np.array(
    all_pred
)

all_prob = np.array(
    all_prob
)

overall_accuracy = accuracy_score(
    all_true,
    all_pred
)

overall_precision = precision_score(
    all_true,
    all_pred,
    zero_division=0
)

overall_recall = recall_score(
    all_true,
    all_pred,
    zero_division=0
)

overall_f1 = f1_score(
    all_true,
    all_pred,
    zero_division=0
)

overall_auc = roc_auc_score(
    all_true,
    all_prob
)

print("OVERALL LOSO RESULTS")
print("=" * 50)

print(
    "Accuracy :",
    overall_accuracy
)

print(
    "Precision:",
    overall_precision
)

print(
    "Recall   :",
    overall_recall
)

print(
    "F1 Score :",
    overall_f1
)

print(
    "ROC-AUC  :",
    overall_auc
)

In [ ]:
cm = confusion_matrix(
    all_true,
    all_pred
)

print(cm)

In [ ]:
plt.figure(
    figsize=(6, 5)
)

plt.imshow(cm)

plt.title(
    "Confusion Matrix - ANN"
)

plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "True Label"
)

plt.xticks(
    [0, 1],
    ["Non-target", "Target"]
)

plt.yticks(
    [0, 1],
    ["Non-target", "Target"]
)

for i in range(2):

    for j in range(2):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()

plt.show()

In [ ]:
print(
    classification_report(
        all_true,
        all_pred,
        target_names=[
            "Non-target",
            "Target"
        ],
        zero_division=0
    )
)

In [ ]:
fpr, tpr, thresholds = roc_curve(
    all_true,
    all_prob
)

roc_auc = auc(
    fpr,
    tpr
)

print(
    "ROC-AUC:",
    roc_auc
)

In [ ]:
plt.figure(
    figsize=(7, 6)
)

plt.plot(
    fpr,
    tpr,
    label=f"ANN (AUC = {roc_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curve - P300 Target Detection"
)

plt.legend()

plt.grid()

plt.show()

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.bar(
    results_df["Subject"],
    results_df["Accuracy"]
)

plt.xlabel(
    "Subject"
)

plt.ylabel(
    "Accuracy"
)

plt.title(
    "LOSO Subject-wise Accuracy"
)

plt.ylim(
    0,
    1
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.show()

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.bar(
    results_df["Subject"],
    results_df["F1"]
)

plt.xlabel(
    "Subject"
)

plt.ylabel(
    "F1 Score"
)

plt.title(
    "LOSO Subject-wise F1 Score"
)

plt.ylim(
    0,
    1
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.show()

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Accuracy"
)

plt.title(
    "ANN Training vs Validation Accuracy"
)

plt.legend()

plt.grid()

plt.show()

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "ANN Training vs Validation Loss"
)

plt.legend()

plt.grid()

plt.show()

In [ ]:
X_all = np.concatenate(
    [
        subject_data[s]["X"]
        for s in loaded_subjects
    ],
    axis=0
)

y_all = np.concatenate(
    [
        subject_data[s]["y"]
        for s in loaded_subjects
    ],
    axis=0
)

print(
    "All subjects X:",
    X_all.shape
)

print(
    "All subjects y:",
    y_all.shape
)

In [ ]:
final_scaler = StandardScaler()

X_all_scaled = final_scaler.fit_transform(
    X_all
)

print(
    "Scaled shape:",
    X_all_scaled.shape
)

In [ ]:
final_model = create_ann(
    X_all_scaled.shape[1]
)

final_model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [ ]:
final_history = final_model.fit(
    X_all_scaled,
    y_all,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.plot(
    final_history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    final_history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title(
    "Final ANN Training Accuracy"
)

plt.legend()
plt.grid()

plt.show()

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.plot(
    final_history.history["loss"],
    label="Training Loss"
)

plt.plot(
    final_history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "Final ANN Training Loss"
)

plt.legend()
plt.grid()

plt.show()

In [ ]:
MODEL_PATH = "/content/p300_ann_model.keras"

final_model.save(
    MODEL_PATH
)

print(
    "✅ Model saved:",
    MODEL_PATH
)

In [ ]:
SCALER_PATH = "/content/p300_scaler.pkl"

joblib.dump(
    final_scaler,
    SCALER_PATH
)

print(
    "✅ Scaler saved:",
    SCALER_PATH
)

In [ ]:
RESULTS_PATH = "/content/LOSO_results.csv"

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print(
    "✅ Results saved:",
    RESULTS_PATH
)

In [ ]:
config = {

    "project":
        PROJECT_NAME,

    "dataset":
        DATASET_ID,

    "subjects":
        loaded_subjects,

    "target_code":
        TARGET_CODE,

    "nontarget_code":
        NONTARGET_CODE,

    "bandpass":
        f"{LOW_FREQ}-{HIGH_FREQ} Hz",

    "notch":
        f"{NOTCH_FREQ} Hz",

    "epoch":
        f"{TMIN}-{TMAX} seconds",

    "baseline":
        "0-0.1 seconds",

    "downsample_factor":
        DOWNSAMPLE_FACTOR,

    "model":
        "Artificial Neural Network",

    "evaluation":
        "Leave-One-Subject-Out",

    "overall_accuracy":
        float(overall_accuracy),

    "overall_precision":
        float(overall_precision),

    "overall_recall":
        float(overall_recall),

    "overall_f1":
        float(overall_f1),

    "overall_roc_auc":
        float(overall_auc)
}

with open(
    "/content/project_config.json",
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )

print("✅ Configuration saved.")

In [ ]:
# CELL 72 — Check saved project files

import os

print("=" * 60)
print("SAVED PROJECT FILES")
print("=" * 60)

files_to_check = [
    "/content/p300_ann_model.keras",
    "/content/p300_scaler.pkl",
    "/content/final_results.json",
    "/content/project_config.json"
]

for file_path in files_to_check:

    if os.path.exists(file_path):

        size_mb = os.path.getsize(file_path) / (1024 * 1024)

        print(
            f"✅ {os.path.basename(file_path)} "
            f"({size_mb:.2f} MB)"
        )

    else:

        print(
            f"❌ Missing: {os.path.basename(file_path)}"
        )

In [ ]:
# ============================================================
# CREATE FINAL RESULTS FILE
# ============================================================

import json
import os

# Check which evaluation variables currently exist
print("Checking evaluation results...")

variables = [
    "overall_accuracy",
    "overall_precision",
    "overall_recall",
    "overall_f1",
    "overall_auc"
]

for var in variables:
    print(var, "->", "FOUND" if var in globals() else "NOT FOUND")

In [ ]:
# ============================================================
# SAVE FINAL EVALUATION RESULTS
# ============================================================

final_results = {
    "Accuracy": float(overall_accuracy),
    "Precision": float(overall_precision),
    "Recall": float(overall_recall),
    "F1_Score": float(overall_f1),
    "ROC_AUC": float(overall_auc)
}

with open("/content/final_results.json", "w") as f:
    json.dump(final_results, f, indent=4)

print("\n" + "="*60)
print("FINAL P300 CLASSIFICATION RESULTS")
print("="*60)

print(f"Accuracy  : {overall_accuracy * 100:.2f}%")
print(f"Precision : {overall_precision * 100:.2f}%")
print(f"Recall    : {overall_recall * 100:.2f}%")
print(f"F1 Score  : {overall_f1 * 100:.2f}%")
print(f"ROC-AUC   : {overall_auc:.4f}")

print("\n✅ final_results.json saved successfully!")

In [ ]:
# ============================================================
# TARGET vs NONTARGET P300 WAVEFORM
# Corrected for flattened EEG data
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Select subject
selected_subject = "sub-1"

X = subject_data[selected_subject]["X"]
y = subject_data[selected_subject]["y"]

print("Subject used:", selected_subject)
print("X shape:", X.shape)

print("Target epochs:", np.sum(y == 1))
print("NonTarget epochs:", np.sum(y == 0))

# ------------------------------------------------------------
# IMPORTANT:
# X has shape:
# (epochs, channels × samples)
#
# Your flattened size is 1664.
# We need to reconstruct:
# (epochs, channels, samples)
# ------------------------------------------------------------

N_CHANNELS = 8
N_SAMPLES = 208

if N_CHANNELS * N_SAMPLES != X.shape[1]:
    raise ValueError(
        f"Channel/sample dimensions do not match. "
        f"{N_CHANNELS} × {N_SAMPLES} != {X.shape[1]}"
    )

# Reshape flattened EEG data
X_3D = X.reshape(
    X.shape[0],
    N_CHANNELS,
    N_SAMPLES
)

print("Reconstructed X shape:", X_3D.shape)

# ------------------------------------------------------------
# Separate Target and NonTarget
# ------------------------------------------------------------

target_epochs = X_3D[y == 1]
nontarget_epochs = X_3D[y == 0]

print("Target shape:", target_epochs.shape)
print("NonTarget shape:", nontarget_epochs.shape)

# ------------------------------------------------------------
# Average across epochs and channels
# ------------------------------------------------------------

target_erp = target_epochs.mean(axis=(0, 1))
nontarget_erp = nontarget_epochs.mean(axis=(0, 1))

print("Target waveform shape:", target_erp.shape)
print("NonTarget waveform shape:", nontarget_erp.shape)

# ------------------------------------------------------------
# Sampling frequency after downsampling
# ------------------------------------------------------------

if 'SFREQ' in globals():
    plot_sfreq = SFREQ / DOWNSAMPLE_FACTOR
else:
    plot_sfreq = 128

# Time axis
times = np.arange(N_SAMPLES) / plot_sfreq

# ------------------------------------------------------------
# Plot Target vs NonTarget
# ------------------------------------------------------------

plt.figure(figsize=(12, 6))

plt.plot(
    times,
    target_erp,
    label="Target (P300)"
)

plt.plot(
    times,
    nontarget_erp,
    label="NonTarget"
)

plt.axvline(
    0,
    linestyle="--",
    label="Stimulus onset"
)

plt.axhline(
    0,
    linestyle=":"
)

plt.xlabel("Time (seconds)")
plt.ylabel("EEG Amplitude")
plt.title("Target vs NonTarget P300 Waveform")

plt.legend()
plt.grid(True)
plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CREATE PREDICTIONS FROM SAVED P300 ANN MODEL
# ============================================================

import os
import numpy as np
import joblib
import tensorflow as tf

MODEL_PATH = "/content/p300_ann_model.keras"
SCALER_PATH = "/content/p300_scaler.pkl"

# ------------------------------------------------------------
# Load saved model
# ------------------------------------------------------------

model = tf.keras.models.load_model(MODEL_PATH)

print("✅ ANN model loaded")

# ------------------------------------------------------------
# Find the test data
# ------------------------------------------------------------

if "X_test_scaled" in globals():

    X_for_prediction = X_test_scaled
    print("Using existing X_test_scaled")

elif "X_test" in globals():

    # Load scaler
    scaler_loaded = joblib.load(SCALER_PATH)

    X_for_prediction = scaler_loaded.transform(X_test)

    print("Using X_test and loaded scaler")

else:
    raise NameError(
        "Neither X_test_scaled nor X_test exists. "
        "We need the test data used during evaluation."
    )

# ------------------------------------------------------------
# Generate predictions
# ------------------------------------------------------------

prediction_probability = model.predict(
    X_for_prediction,
    verbose=0
).ravel()

# Convert probability to Target / NonTarget
y_pred = (prediction_probability >= 0.5).astype(int)

print("✅ Predictions generated")
print("Number of predictions:", len(y_pred))
print("Number of true labels:", len(y_test))

print("\nPredicted classes:")
print("NonTarget:", np.sum(y_pred == 0))
print("Target   :", np.sum(y_pred == 1))

In [ ]:
# ============================================================
# FINAL CONFUSION MATRIX
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1]
)

print("\n" + "="*55)
print("FINAL P300 ANN CONFUSION MATRIX")
print("="*55)

print("\n                 Predicted")
print("              NonTarget   Target")
print(f"Actual NonTarget   {cm[0,0]:6d}    {cm[0,1]:6d}")
print(f"Actual Target      {cm[1,0]:6d}    {cm[1,1]:6d}")

# Plot
fig, ax = plt.subplots(figsize=(7, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["NonTarget", "Target"]
)

disp.plot(
    ax=ax,
    values_format="d"
)

ax.set_title("P300 ANN Confusion Matrix")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("Actual Label")

plt.tight_layout()
plt.show()

In [ ]:
# CELL 73 — Display saved configuration

with open(
    "/content/project_config.json",
    "r"
) as f:

    saved_config = json.load(f)

print("=" * 60)
print("PROJECT CONFIGURATION")
print("=" * 60)

for key, value in saved_config.items():

    print(
        f"{key}: {value}"
    )

In [ ]:
# CELL 74 — Download trained ANN model

from google.colab import files

files.download(
    "/content/p300_ann_model.keras"
)

In [ ]:
# 4 Smart Applications

applications = {
    1: "LIGHT",
    2: "FAN",
    3: "TV",
    4: "AC"
}

print("Available Applications:")
for number, app in applications.items():
    print(f"{number}. {app}")

In [ ]:
# Initial state of all applications

app_state = {
    "LIGHT": "OFF",
    "FAN": "OFF",
    "TV": "OFF",
    "AC": "OFF"
}

print("Initial Application States:")
for app, state in app_state.items():
    print(f"{app}: {state}")

In [ ]:
# Application selected by the P300 command

selected_application = 1

selected_app = applications[selected_application]

print("P300 Target Detected")
print("Selected Application:", selected_app)

In [ ]:
# Turn the selected application ON

app_state[selected_app] = "ON"

print("\nApplication Command:")
print(selected_app, "-> ON")

In [ ]:
print("=" * 45)
print("       P300 SMART APPLICATION CONTROL")
print("=" * 45)

print("\nDetected P300 Target : YES")
print("Selected Application : " + selected_app)
print("Command              : " + app_state[selected_app])

print("\n" + "-" * 45)
print("FINAL OUTPUT")
print("-" * 45)

print("LIGHT : " + app_state["LIGHT"])
print("FAN   : " + app_state["FAN"])
print("TV    : " + app_state["TV"])
print("AC    : " + app_state["AC"])

print("=" * 45)